In [33]:
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.utils import to_categorical


In [26]:
# Load the dataset
df = pd.read_csv('./SP500_Historical_Data.csv')

# Create the target variable: 1 if the next day's closing price is higher than today's, else 0. For the one hot encoding.
df["Next Day Close"] = (df.groupby("Ticker")["Close"].shift(-1))
df["Target"] = (df["Next Day Close"] > df["Close"]).astype(int)
df = df.dropna()

# Feature Selection: Drop columns that are not needed for modeling.
df.drop(columns=["Ticker", "Date", "Adj Close", "Next Day Close"], inplace=True)

df.iloc[6568:6578]

,Open,High,Low,Close,Volume,Target
6568,125.60,127.77,125.02,125.81,1527900,0
6569,125.87,125.91,122.99,123.87,2110500,1
6570,123.64,126.99,123.05,126.94,1370400,0
6571,126.01,126.36,123.66,126.34,1892900,0
6573,19.84,20.17,18.01,18.19,961200,1
6574,18.19,19.35,18.10,19.33,5747900,0
6575,19.23,19.40,18.95,19.05,1078200,1
6576,19.10,19.84,19.02,19.81,3123300,1
6577,19.70,20.50,19.70,20.27,1057900,1
6578,20.21,21.21,20.21,20.89,1768800,1


In [ ]:
# Extract values of features and target variable
X = df[["Open", "High", "Low", "Close", "Volume"]].values
y = df["Target"].values

# One-hot encode the target variable
y_encoded = to_categorical(y)

# Normalize the features using Min-Max Scaling (0-1 range)
scaler = MinMaxScaler()
x_scaled = scaler.fit_transform(X)

# Time Series Split Test
tscv = TimeSeriesSplit(n_splits=5)
for i, (train_index, test_index) in enumerate(tscv.split(x_scaled)):
    X_train, X_test = x_scaled[train_index], x_scaled[test_index]
    y_train, y_test = y_encoded[train_index], y_encoded[test_index]
    print(f"Fold {i+1}")
    print(f"Train indices: {train_index}")
    print(f"Test indices: {test_index}")
    print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
    print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")
    print("-" * 50)

Fold 1
Train indices: [     0      1      2 ... 450511 450512 450513]
Test indices: [450514 450515 450516 ... 901020 901021 901022]
X_train shape: (450514, 5), y_train shape: (450514, 2)
X_test shape: (450509, 5), y_test shape: (450509, 2)
--------------------------------------------------
Fold 2
Train indices: [     0      1      2 ... 901020 901021 901022]
Test indices: [ 901023  901024  901025 ... 1351529 1351530 1351531]
X_train shape: (901023, 5), y_train shape: (901023, 2)
X_test shape: (450509, 5), y_test shape: (450509, 2)
--------------------------------------------------
Fold 3
Train indices: [      0       1       2 ... 1351529 1351530 1351531]
Test indices: [1351532 1351533 1351534 ... 1802038 1802039 1802040]
X_train shape: (1351532, 5), y_train shape: (1351532, 2)
X_test shape: (450509, 5), y_test shape: (450509, 2)
--------------------------------------------------
Fold 4
Train indices: [      0       1       2 ... 1802038 1802039 1802040]
Test indices: [1802041 1802042 

In [59]:
model = Sequential()
# Input Layer with 5 features
model.add(Dense(64, input_dim=5, activation='relu'))
# 2 Hidden Layers
model.add(Dense(32, activation='relu'))
model.add(Dense(16, activation='relu'))
# Output Layer with 2 classes (e.g. [0, 1], [0.33, 0.67], etc.)
model.add(Dense(2, activation='softmax'))

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()


c:\Users\gamma\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_8 (Dense)                 │ (None, 64)             │           384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 2)              │            34 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,026 (11.82 KB)

 Trainable params: 3,026 (11.82 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Using the last 20% of data as the test set // Not TimeSeriesSplit, but a simple split for testing
split_idx = int(len(x_scaled) * 0.8)
X_train, X_test = x_scaled[:split_idx], x_scaled[split_idx:]
y_train, y_test = y_encoded[:split_idx], y_encoded[split_idx:]
print(f"Final Train set shape: X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"Final Test set shape: X_test: {X_test.shape}, y_test: {y_test.shape}")

Final Train set shape: X_train: (2162447, 5), y_train: (2162447, 2)
Final Test set shape: X_test: (540612, 5), y_test: (540612, 2)


In [63]:
import time 
tic = time.time()
history = model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), verbose=1)
toc = time.time()

Epoch 1/100
67577/67577 ━━━━━━━━━━━━━━━━━━━━ 153s 2ms/step - accuracy: 0.5131 - loss: 0.6927 - val_accuracy: 0.5131 - val_loss: 0.6927
Epoch 2/100
67577/67577 ━━━━━━━━━━━━━━━━━━━━ 147s 2ms/step - accuracy: 0.5138 - loss: 0.6926 - val_accuracy: 0.5107 - val_loss: 0.6929
Epoch 3/100
67577/67577 ━━━━━━━━━━━━━━━━━━━━ 145s 2ms/step - accuracy: 0.5142 - loss: 0.6925 - val_accuracy: 0.5127 - val_loss: 0.6925
Epoch 4/100
67577/67577 ━━━━━━━━━━━━━━━━━━━━ 146s 2ms/step - accuracy: 0.5141 - loss: 0.6924 - val_accuracy: 0.5122 - val_loss: 0.6926
Epoch 5/100
67577/67577 ━━━━━━━━━━━━━━━━━━━━ 146s 2ms/step - accuracy: 0.5140 - loss: 0.6924 - val_accuracy: 0.5132 - val_loss: 0.6926
Epoch 6/100
67577/67577 ━━━━━━━━━━━━━━━━━━━━ 145s 2ms/step - accuracy: 0.5139 - loss: 0.6924 - val_accuracy: 0.5108 - val_loss: 0.6926
Epoch 7/100
67577/67577 ━━━━━━━━━━━━━━━━━━━━ 145s 2ms/step - accuracy: 0.5141 - loss: 0.6923 - val_accuracy: 0.5105 - val_loss: 0.6924
Epoch 8/100
67577/67577 ━━━━━━━━━━━━━━━━━━━━ 161s 2ms/s

In [67]:
print("Time taken for training: {:.2f} hours".format((toc - tic)/3600))

Time taken for training: 9.65 hours
